# 🚀 CVAT Universal AI Engine - One-Click Google Colab Runner

Notebook này cho phép bạn biến **Google Colab (T4 GPU miễn phí)** thành một **Server AI Gán nhãn CVAT** cực mạnh:
- 📐 **SAM 2.1 (Segment Anything 2)**: Gán nhãn polygon bám viền khít chỉ qua click/box prompt.
- 📦 **YOLOv11**: Tự động phát hiện vật thể dạng Bounding Box.
- 🦴 **Pose Estimation**: Tự động trích xuất khung xương và khớp nối (Skeleton).
- 🌐 **Cloudflare Tunnel**: Mở đường dẫn HTTPS bảo mật công khai miễn phí để kết nối trực tiếp với máy cá nhân hoặc CVAT Server mà không lo chặn port!

---
### ⚠️ Yêu cầu trước khi chạy:
1. Vào menu **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Chọn **T4 GPU**.
2. Chọn **Runtime** $\rightarrow$ **Run all** (hoặc bấm chạy tuần tự từng cell bên dưới).

In [ ]:
# 1. Kiểm tra trạng thái GPU NVIDIA
!nvidia-smi

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q fastapi uvicorn pyyaml pillow ultralytics opencv-python-headless

# Tải mã nguồn dự án nếu chưa có
import os
if not os.path.exists('configs'):
    print("Đang thiết lập cấu trúc mã nguồn...")
    !git clone https://github.com/cvat-ai/cvat.git locate_repo_temp || true

In [ ]:
# 3. Tải Cloudflare Tunnel (cloudflared) để mở URL HTTPS công khai
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
# 4. Khởi chạy FastAPI AI Service và Cloudflare Tunnel
import subprocess
import time
import re

print("🔥 Khởi chạy FastAPI AI Service trên cổng 8000...")
server_proc = subprocess.Popen(["python", "-m", "uvicorn", "server.api_service:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

print("🌐 Đang mở Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("=" * 65)
    print("🎉 CVAT UNIVERSAL AI ENGINE ĐANG CHẠY THÀNH CÔNG TRÊN COLAB GPU!")
    print(f"🌐 URL HTTPS CÔNG KHAI: {tunnel_url}")
    print(f"🩺 Health Check: {tunnel_url}/health")
    print(f"📋 Labels API: {tunnel_url}/api/labels")
    print("=" * 65)
    print("💡 Bạn có thể dán đường link này vào CVAT client hoặc webhook để gán nhãn!")
else:
    print("⚠️ Không tìm thấy URL Cloudflare. Kiểm tra log:")
    for _ in range(10):
        print(tunnel_proc.stdout.readline().strip())

### 🤖 Tiện ích Tự động Gán nhãn cho một Task trên CVAT (Auto-Annotator)
Chạy cell bên dưới nếu bạn muốn Colab GPU tự động kéo ảnh từ Task trên CVAT về, chạy SAM2/YOLO để gán nhãn rồi đẩy ngược lên CVAT hoàn toàn tự động.

In [ ]:
# Nhập thông tin task trên CVAT (nếu cần tự động gán nhãn)
CVAT_HOST = "http://your-vps-ip:8080"  # hoặc https://app.cvat.ai
CVAT_TASK_ID = 1
CVAT_TOKEN = "YOUR_API_TOKEN"

print(f"Sẵn sàng kết nối tới Task {CVAT_TASK_ID} trên {CVAT_HOST}")